# 04 - Analisis ACF / PACF (HICP Eurozona)

Estimacion visual de los ordenes p, q (ARIMA) y P, Q (SARIMA).

**Reglas Box-Jenkins:**
- ACF decaimiento exponencial + PACF corte en lag k -> AR(k)
- ACF corte en lag k + PACF decaimiento -> MA(k)
- Picos en lags 12, 24, 36 -> ordenes estacionales P, Q

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

NOTEBOOK_DIR = Path(r"c:/Users/usuario/OneDrive/Documentos/GitHub/tfg-ipc-mcp/tfg-forecasting/02_eda")
ROOT = NOTEBOOK_DIR.parent
MONOREPO = ROOT.parent
sys.path.insert(0, str(MONOREPO))
from shared.constants import DATE_TRAIN_END
plt.rcParams.update({"figure.figsize": (14, 4), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
df = pd.read_parquet(ROOT / "data" / "processed" / "hicp_europe_index.parquet")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
train = df.loc[:DATE_TRAIN_END]
y = train["hicp_index"]
y.index.freq = "MS"

y_diff1    = y.diff().dropna()
y_diff12   = y.diff(12).dropna()
y_diff1_12 = y.diff().diff(12).dropna()
print(f"Nivel: {len(y)} | diff(1): {len(y_diff1)} | diff(12): {len(y_diff12)} | diff(1,12): {len(y_diff1_12)}")

## 1. ACF / PACF - Serie en nivel

Esperamos ACF con decaimiento muy lento (no estacionaria).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y, lags=48, ax=axes[0], title="ACF - HICP nivel")
plot_pacf(y, lags=48, ax=axes[1], title="PACF - HICP nivel", method="ywm")
plt.tight_layout(); plt.show()

## 2. ACF / PACF - Primera diferencia (d=1)

Buscamos ordenes p y q. Picos en multiplos de 12 indican componente estacional.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_diff1, lags=48, ax=axes[0], title="ACF - diff(1)")
plot_pacf(y_diff1, lags=48, ax=axes[1], title="PACF - diff(1)", method="ywm")
for ax in axes:
    for lag in [12, 24, 36, 48]:
        ax.axvline(lag, color="red", linestyle=":", alpha=0.3)
plt.tight_layout(); plt.show()

## 3. ACF / PACF - Diferencia estacional (D=1, lag 12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_diff12, lags=48, ax=axes[0], title="ACF - diff(12)")
plot_pacf(y_diff12, lags=48, ax=axes[1], title="PACF - diff(12)", method="ywm")
for ax in axes:
    for lag in [12, 24, 36, 48]:
        ax.axvline(lag, color="red", linestyle=":", alpha=0.3)
plt.tight_layout(); plt.show()

## 4. ACF / PACF - Doble diferencia (d=1, D=1)

Serie sobre la que SARIMA opera internamente. Determinamos los ordenes finales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_diff1_12, lags=48, ax=axes[0], title="ACF - diff(1) + diff(12)")
plot_pacf(y_diff1_12, lags=48, ax=axes[1], title="PACF - diff(1) + diff(12)", method="ywm")
for ax in axes:
    for lag in [12, 24, 36, 48]:
        ax.axvline(lag, color="red", linestyle=":", alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Valores numericos en lags clave

In [ ]:
lags_key = [1, 2, 3, 6, 12, 24, 36]
n  = len(y_diff1_12)
ci = 1.96 / np.sqrt(n)

acf_vals  = acf(y_diff1_12, nlags=max(lags_key))
pacf_vals = pacf(y_diff1_12, nlags=max(lags_key), method="ywm")

print(f"IC 95%: +/- {ci:.4f}")
print(f"{'Lag':>4} {'ACF':>10} {'PACF':>10} {'ACF sig':>10} {'PACF sig':>10}")
print("-" * 50)
for lag in lags_key:
    a, p = acf_vals[lag], pacf_vals[lag]
    print(f"{lag:4d} {a:10.4f} {p:10.4f} {'*' if abs(a)>ci else '':>10} {'*' if abs(p)>ci else '':>10}")

## 6. Estimacion de ordenes candidatos

In [ ]:
acf_full  = acf(y_diff1_12, nlags=36)
pacf_full = pacf(y_diff1_12, nlags=36, method="ywm")
n  = len(y_diff1_12)
ci = 1.96 / np.sqrt(n)

acf_sig_reg  = [i for i in range(1, 7) if abs(acf_full[i])  > ci]
pacf_sig_reg = [i for i in range(1, 7) if abs(pacf_full[i]) > ci]
seas_lags = [12, 24, 36]
acf_sig_sea  = [i for i in seas_lags if abs(acf_full[i])  > ci]
pacf_sig_sea = [i for i in seas_lags if abs(pacf_full[i]) > ci]

print("=" * 60)
print("ORDENES CANDIDATOS - HICP Eurozona")
print("=" * 60)
print(f"ACF  sig regulares:    {acf_sig_reg}")
print(f"PACF sig regulares:    {pacf_sig_reg}")
print(f"ACF  sig estacionales: {acf_sig_sea}")
print(f"PACF sig estacionales: {pacf_sig_sea}")
print()
print(f"d = 1 (confirmar con tests de estacionariedad)")
print(f"D = 1, m = 12")
print(f"p candidatos: {pacf_sig_reg if pacf_sig_reg else [0,1,2,3]}")
print(f"q candidatos: {acf_sig_reg  if acf_sig_reg  else [0,1,2]}")
print(f"P candidatos: {[l//12 for l in pacf_sig_sea] if pacf_sig_sea else [0,1]}")
print(f"Q candidatos: {[l//12 for l in acf_sig_sea]  if acf_sig_sea  else [0,1]}")

## 7. Ljung-Box - Verificacion de ruido blanco

In [ ]:
lb = acorr_ljungbox(y_diff1_12, lags=[6, 12, 24], return_df=True)
print("Test de Ljung-Box (H0: no autocorrelacion hasta lag k):")
print(lb)
print("
Si p > 0.05, la diferenciacion ha eliminado la autocorrelacion.")

## 8. Conclusion

**Resumen de ordenes para SARIMA (HICP Eurozona):**

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| d | 1 | Tests ADF/KPSS/PP (notebook 02) |
| D | 1 | Estacionalidad significativa (notebook 03) |
| m | 12 | Frecuencia mensual |
| p | TBD | Corte en PACF regular |
| q | TBD | Corte en ACF regular |
| P | TBD | Corte en PACF estacional |
| Q | TBD | Corte en ACF estacional |

*Completar tras ejecucion. Guian la busqueda de auto_arima.*